In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_care_teams;
SET search_path TO qdp_care_teams;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata (
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata
(
  supplier,
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
)
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.care_teams',
  '0.1',
  'care_teams.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_care_team table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_care_team CASCADE;

CREATE TABLE hub_care_team (
    care_team_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    care_team_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE hub_care_team ADD CONSTRAINT pk_careteam_hubcareteam_careteamid PRIMARY KEY (care_team_id);

COMMENT ON COLUMN hub_care_team.care_team_id IS 'Care Team Identity';




-- -----------------------------------------------------------
-- create sat_care_team table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_team;

CREATE TABLE sat_care_team (
    sat_care_team_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    care_team_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 1,
	data_source_code VARCHAR,
	data_source_concept_id BIGINT,
	data_source_raw_code VARCHAR,
	data_source_raw_concept_id BIGINT,
	status_code VARCHAR,
	status_concept_id BIGINT,
	status_raw_code VARCHAR,
	status_raw_concept_id BIGINT,
	name VARCHAR,
	subject_party_id BIGINT,
	participant_party_id BIGINT,
	managing_organisation_party_id BIGINT,
	note TEXT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_team ADD CONSTRAINT pk_careteam_satcareteam_satcareteamid PRIMARY KEY (sat_care_team_id);
ALTER TABLE sat_care_team ADD CONSTRAINT fk_careteam_satcareteam_hubcareteam_careteamid FOREIGN KEY (care_team_id) REFERENCES hub_care_team(care_team_id);

COMMENT ON COLUMN sat_care_team.care_team_id IS 'Care Team Identity';
COMMENT ON COLUMN sat_care_team.sat_care_team_id IS 'Care Team Identity Sat';
COMMENT ON COLUMN sat_care_team.status_code IS 'Current status of the Care Team. E.g. proposed, active, suspended, inactive, entered-in-error. See https://build.fhir.org/valueset-care-team-status.html';
COMMENT ON COLUMN sat_care_team.name IS 'The name of the Care Team, such as crisis assessment team. See https://build.fhir.org/careteam-definitions.html#CareTeam.name';
COMMENT ON COLUMN sat_care_team.subject_party_id IS 'Unique identifier of the Patient or Group the Care Team is for. See https://build.fhir.org/careteam-definitions.html#CareTeam.subject';
COMMENT ON COLUMN sat_care_team.participant_party_id IS 'Unique Identifier for the Party Group containing the list of Practitioners who are members of the Care Team. See https://build.fhir.org/careteam-definitions.html#CareTeam.participant';
COMMENT ON COLUMN sat_care_team.managing_organisation_party_id IS 'Unique identifier of the Organisation responsible for the care team. See https://build.fhir.org/careteam-definitions.html#CareTeam.managingOrganization';
COMMENT ON COLUMN sat_care_team.note IS 'Comments made about the CareTeam. See https://build.fhir.org/careteam-definitions.html#CareTeam.note';
COMMENT ON COLUMN sat_care_team.period_start IS 'The start date and time indicating when the care team was active. See https://build.fhir.org/careteam-definitions.html#CareTeam.period';
COMMENT ON COLUMN sat_care_team.period_end IS 'The end date and time indicating when the care team was no longer active. See https://build.fhir.org/careteam-definitions.html#CareTeam.period';



-- -----------------------------------------------------------
-- create sat_care_team_category table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_team_category;

CREATE TABLE sat_care_team_category (
	care_team_category_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	care_team_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	category_code VARCHAR,
	category_concept_id BIGINT,
	category_raw_code VARCHAR,
	category_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_team_category ADD CONSTRAINT pk_careteam_satcareteamcategory_careteamcategoryid PRIMARY KEY (care_team_category_id);
ALTER TABLE sat_care_team_category ADD CONSTRAINT fk_careteam_satcareteamcategory_hubcareteam_careteamid FOREIGN KEY (care_team_id) REFERENCES hub_care_team(care_team_id);

COMMENT ON COLUMN sat_care_team_category.care_team_id IS 'Care Team Identity';
COMMENT ON COLUMN sat_care_team_category.care_team_category_id IS 'Care Team Category Identity';
COMMENT ON COLUMN sat_care_team_category.category_code IS 'Type of Care Team. E.g. event-focused, encounter-focused, episode of care-focused, condition-focused, longitudinal care-coordination, HCBS-focused, clinical research-focused, public health-focused. See http://hl7.org/fhir/ValueSet/care-team-category';
COMMENT ON COLUMN sat_care_team_category.rank IS 'The recommended order the Care Team Category Codes should be presented';



-- -----------------------------------------------------------
-- create sat_care_team_reason table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_team_reason;

CREATE TABLE sat_care_team_reason (
	care_team_reason_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	care_team_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	reason_code VARCHAR,
	reason_concept_id BIGINT,
	reason_raw_code VARCHAR,
	reason_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_team_reason ADD CONSTRAINT pk_careteam_satcareteamreason_careteamreasonid PRIMARY KEY (care_team_reason_id);
ALTER TABLE sat_care_team_reason ADD CONSTRAINT fk_careteam_satcareteamreason_hubcareteam_careteamid FOREIGN KEY (care_team_id) REFERENCES hub_care_team(care_team_id);

COMMENT ON COLUMN sat_care_team_reason.care_team_id IS 'Care Team Identity';
COMMENT ON COLUMN sat_care_team_reason.care_team_reason_id IS 'Care Team Reason Identity';
COMMENT ON COLUMN sat_care_team_reason.reason_code IS 'Why the care team exists, coded as clinical findings (SNOMED CT). See https://build.fhir.org/valueset-clinical-findings.html';
COMMENT ON COLUMN sat_care_team_reason.rank IS 'The recommended order the Care Team Reason Codes should be presented';



-- -----------------------------------------------------------
-- create view_care_teams view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_care_teams AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.care_team_id,
	h.care_team_entity_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
  s.sat_care_team_id,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.name,
  s.subject_party_id,
  s.participant_party_id,
  s.managing_organisation_party_id,
  s.note,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_care_teams.hub_care_team h
    JOIN budd.qdp_care_teams.sat_care_team s
      ON h.care_team_id = s.care_team_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_care_teams;


-- -----------------------------------------------------------
-- create view_care_team_categories view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_care_team_categories AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.care_team_id,
	h.care_team_entity_id,
  s.name AS care_team_name,
  s.status_code AS care_team_status_code,
	ctc.care_team_category_id,
	ctc.category_code,
	ctc.category_concept_id,
	ctc.category_raw_code,
	ctc.category_raw_concept_id,
	ctc.rank AS category_rank,
	ctc.period_start AS category_period_start,
	ctc.period_end AS category_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_care_teams.hub_care_team h
    JOIN budd.qdp_care_teams.sat_care_team s
      ON h.care_team_id = s.care_team_id
    JOIN budd.qdp_care_teams.sat_care_team_category ctc
      ON h.care_team_id = ctc.care_team_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_care_team_categories;


-- -----------------------------------------------------------
-- create view_care_team_reasons view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_care_team_reasons AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.care_team_id,
	h.care_team_entity_id,
  s.name AS care_team_name,
  s.status_code AS care_team_status_code,
	ctr.care_team_reason_id,
	ctr.reason_code,
	ctr.reason_concept_id,
	ctr.reason_raw_code,
	ctr.reason_raw_concept_id,
	ctr.rank AS reason_rank,
	ctr.period_start AS reason_period_start,
	ctr.period_end AS reason_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_care_teams.hub_care_team h
    JOIN budd.qdp_care_teams.sat_care_team s
      ON h.care_team_id = s.care_team_id
    JOIN budd.qdp_care_teams.sat_care_team_reason ctr
      ON h.care_team_id = ctr.care_team_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_care_team_reasons;


/**

**/
